In [ ]:
#Requirements Installation

%pip install openpyxl

In [ ]:
#Imports

from pyspark.sql import SparkSession
from datetime import datetime
import importlib.util
import sys
import os
import yaml

spark = SparkSession.builder.getOrCreate()

In [ ]:
#Path Definitions

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()

NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
SCRIPTS_DIR  = os.path.abspath(f"{NOTEBOOK_DIR}/../scripts")
INGESTION_DIR = os.path.abspath(f"{NOTEBOOK_DIR}/../drugdev/ingestion_framework")

print("Notebook Directory:", NOTEBOOK_DIR)
print("Scripts Directory :", SCRIPTS_DIR)
print("Ingestion Directory:", INGESTION_DIR)

In [ ]:
# ── Standalone Environment Variables ─────────────────────────────────────────
# Set all values here — no widget prompts required.
# Update each value to match your target Databricks environment.

RAW_BUCKET       = "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw"
STG_BUCKET       = ""                                     # staging bucket if applicable
CATALOG          = "dev-drugdev_da-koios-catalog"                             # Unity Catalog name
BRONZE_SCHEMA    = "dev_drugdev_bronze"
SILVER_SCHEMA    = "dev_drugdev_silver"
GOLD_SCHEMA      = "dev_drugdev_gold"
METADATA_CATALOG = "dev-drugdev_da-koios-catalog"                             # catalog hosting metadata tables
REGISTRY_SCHEMA  = "dev_drugdev_common"
METADATA_SCHEMA  = "dev_drugdev_common"
RUN_DATE         = datetime.now().strftime("%Y%m%d")      # or set explicitly e.g. "20260718"
ENVIRONMENT      = "dev"
SOURCE_BUCKET    = "exelixis-clearlake-daplex-dev-us-west-2-441447966705-raw"
SIMULATION_TYPE  = ""                                     # e.g. "dry_run" or leave empty

# ── Derived aliases (keep these as-is) ───────────────────────────────────────
run_date        = RUN_DATE
simulation_type = SIMULATION_TYPE
environment     = ENVIRONMENT
source_bucket   = SOURCE_BUCKET

# ── Propagate to Spark config ─────────────────────────────────────────────────
spark.conf.set("drugdev.environment",    ENVIRONMENT)
spark.conf.set("drugdev.raw_bucket",     RAW_BUCKET)
spark.conf.set("drugdev.stg_bucket",     STG_BUCKET)
spark.conf.set("drugdev.CATALOG",        CATALOG)
spark.conf.set("drugdev.BRONZE_SCHEMA",  BRONZE_SCHEMA)
spark.conf.set("drugdev.SILVER_SCHEMA",  SILVER_SCHEMA)
spark.conf.set("drugdev.GOLD_SCHEMA",    GOLD_SCHEMA)
spark.conf.set("drugdev.simulation_type", SIMULATION_TYPE)
spark.conf.set("drugdev.METADATA_CATALOG", METADATA_CATALOG)
spark.conf.set("drugdev.METADATA_SCHEMA",  METADATA_SCHEMA)
spark.conf.set("drugdev.REGISTRY_SCHEMA",  REGISTRY_SCHEMA)
spark.conf.set("drugdev.source_bucket",    SOURCE_BUCKET)
spark.conf.set("drugdev.YML_CONFIG_PATH",
               f"{INGESTION_DIR}/configs/drugdev_config.yaml")
spark.conf.set("drugdev.SCHEMA_REGISTRY_PATH",
               f"{INGESTION_DIR}/ingestion_engine/schema_registry.py")
spark.conf.set("drugdev.RUN_DATE", run_date)

print("**Simulation Type** =", simulation_type)
print("RUN_DATE            =", run_date)

In [ ]:
#Import Helper Function

def import_from_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)

    # Inject dbutils into the module
    mod.dbutils = dbutils

    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

In [ ]:
# Load drugdev config and schema registry

config_path = f"{INGESTION_DIR}/configs/drugdev_config.yaml"
with open(config_path, "r") as f:
    drugdev_config = yaml.safe_load(f)

schema_reg = import_from_path(
    "schema_registry",
    f"{INGESTION_DIR}/ingestion_engine/schema_registry.py"
)

In [ ]:
# Create metadata tables and volumes (idempotent)

METADATA_CATALOG = spark.conf.get("drugdev.METADATA_CATALOG")
REGISTRY_SCHEMA  = spark.conf.get("drugdev.REGISTRY_SCHEMA")
METADATA_SCHEMA  = spark.conf.get("drugdev.METADATA_SCHEMA")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_registry (
  dataset_id STRING NOT NULL,
  domain_name STRING,
  data_product_name STRING,
  dataset_name STRING,
  dataset_version STRING,
  frequency STRING,
  owner_team STRING,
  owner_email STRING,
  criticality STRING,
  contains_pii BOOLEAN,
  data_classification STRING,
  lifecycle_status STRING,
  retention_days INT,
  created_at TIMESTAMP,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_config (
  config_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  source_bucket STRING,
  source_path STRING,
  file_format STRING,
  delimiter STRING,
  file_encoding STRING,
  load_type STRING,
  file_name STRING,
  row_tag STRING,
  ingestion_mode STRING,
  primary_keys STRING,
  watermark_column STRING,
  checkpoint_location STRING,
  schema_location STRING,
  schema_strategy STRING,
  optimize_write BOOLEAN,
  auto_compact BOOLEAN,
  is_active BOOLEAN
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_tags (
  tag_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  tag_key STRING,
  tag_value STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{REGISTRY_SCHEMA}.dataset_dependencies (
  dependency_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  depends_on_dataset_id STRING,
  dependency_type STRING,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.ingestion_runtime_state (
  runtime_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  environment STRING,
  last_run_status STRING,
  records_ingested BIGINT,
  files_processed INT,
  last_run_start_time TIMESTAMP,
  last_run_end_time TIMESTAMP,
  failure_reason STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.dq_sla_config (
  dq_id STRING NOT NULL,
  dataset_id STRING NOT NULL,
  dq_enabled BOOLEAN,
  rule_set_name STRING,
  row_count_min BIGINT,
  row_count_max BIGINT,
  freshness_minutes INT,
  null_threshold_pct DOUBLE,
  duplicate_threshold_pct DOUBLE,
  anomaly_detection_enabled BOOLEAN,
  fail_action STRING,
  alert_channel STRING,
  escalation_contact STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schema_registry (
  dataset_id STRING NOT NULL,
  schema_json STRING NOT NULL,
  version INT,
  is_active BOOLEAN,
  created_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.schemas")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{METADATA_CATALOG}`.{METADATA_SCHEMA}.checkpoints")

In [ ]:
#Run metadata loader

metadata_loader = import_from_path(
    "metadata_loader",
    f"{INGESTION_DIR}/metadata_service/metadata_loader.py"
)

metadata_loader.main()

In [ ]:
# Autoloader engine

RAW_BUCKET = spark.conf.get("drugdev.raw_bucket")
autoloader_engine = import_from_path(
    "autoloader_engine",
    f"{INGESTION_DIR}/ingestion_engine/autoloader_engine.py"
)
autoloader_engine.main(simulation_type)